# Full Decoder-Only Transformer — Exercise

The capstone. Assemble everything into a working transformer language model.

```
token_ids → token_embedding
  └→ for each of N transformer blocks:
       └→ x = x + MultiHeadAttention(RMSNorm(x))
       └→ x = x + SwiGLU(RMSNorm(x))
  └→ final_norm = RMSNorm(x)
  └→ logits = LM_head(final_norm)        # → (batch, seq, vocab_size)
```

Two TODOs: a `TransformerBlock` (one layer of pre-norm attention + FFN with residuals), then a full `TransformerLM` model that stacks N blocks.

**Pre-requisites:** complete `04-attention-mechanisms`, `01_swiglu_ffn_exercise`, `02_rmsnorm_exercise`.

## Setup

In [ ]:
# TODO: imports — torch, torch.nn as nn
# TODO: paste in (or import) your MultiHeadAttention, SwiGLU, RMSNorm classes

## TODO 1 — `TransformerBlock`

One pre-norm transformer layer:
```
x = x + MultiHeadAttention(RMSNorm(x))    # attention sublayer
x = x + SwiGLU(RMSNorm(x))                 # FFN sublayer
return x
```

Note: each sublayer has its own RMSNorm (two RMSNorms per block total).

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        # TODO 1a: norm1 = RMSNorm(d_model)
        # TODO 1b: attention = MultiHeadAttention(d_model, n_heads, causal=True)
        # TODO 1c: norm2 = RMSNorm(d_model)
        # TODO 1d: ffn = SwiGLU(d_model)
        pass

    def forward(self, x):
        # TODO 1e: x = x + self.attention(self.norm1(x))
        # TODO 1f: x = x + self.ffn(self.norm2(x))
        # TODO 1g: return x
        pass

## TODO 2 — `TransformerLM`

The full model:
```
token_embedding (vocab_size, d_model)
N × TransformerBlock
final_norm = RMSNorm
LM head (linear, d_model → vocab_size, no bias)
```

**No positional encoding here** — assume RoPE is applied inside the attention (that's how modern LLMs do it). For this exercise, you can skip positional info entirely and the model will still work as a code skeleton.

**Don't tie embedding weights** — modern large-scale convention. (Smaller models often tie, but skip the optimization here.)

In [ ]:
class TransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 512,
        n_heads: int = 8,
        n_layers: int = 6,
    ):
        super().__init__()
        # TODO 2a: token_embedding = nn.Embedding(vocab_size, d_model)
        # TODO 2b: blocks = nn.ModuleList([TransformerBlock(...) for _ in range(n_layers)])
        # TODO 2c: final_norm = RMSNorm(d_model)
        # TODO 2d: lm_head = nn.Linear(d_model, vocab_size, bias=False)
        pass

    def forward(self, token_ids):
        # token_ids: (batch, seq) integer tensor
        # TODO 2e: x = self.token_embedding(token_ids)   → (batch, seq, d_model)
        # TODO 2f: for block in self.blocks: x = block(x)
        # TODO 2g: x = self.final_norm(x)
        # TODO 2h: logits = self.lm_head(x)              → (batch, seq, vocab_size)
        # TODO 2i: return logits
        pass

In [ ]:
# Sanity check:
#   - model = TransformerLM(vocab_size=1000, d_model=128, n_heads=4, n_layers=2)
#   - tokens = torch.randint(0, 1000, (2, 16))
#   - logits = model(tokens)
#   - logits.shape == (2, 16, 1000)
#   - logits.sum().backward()  # gradients flow

## TODO 3 — Verify causality end-to-end

The model is decoder-only. Position 5's logits should only depend on tokens 0..5, NOT on tokens 6..15.

In [ ]:
# TODO 3:
#   - tokens_a = torch.randint(0, 1000, (1, 16))
#   - tokens_b = tokens_a.clone()
#   - tokens_b[0, 10:] = 999       # change positions 10..15
#   - logits_a = model(tokens_a)
#   - logits_b = model(tokens_b)
#   - Assert torch.allclose(logits_a[0, :10], logits_b[0, :10], atol=1e-5)
#   - This is the property that makes parallel teacher-forcing training valid

## TODO 4 — Generation (greedy)

Given a prompt of token IDs, generate `max_new_tokens` more tokens by repeated forward passes. Greedy decoding (argmax of logits).

In [ ]:
# TODO 4: implement generate(model, prompt, max_new_tokens)
#   for _ in range(max_new_tokens):
#       logits = model(prompt)             # (1, seq, vocab)
#       next_token = logits[:, -1].argmax(-1, keepdim=True)
#       prompt = torch.cat([prompt, next_token], dim=1)
#   return prompt

## Run the tests

In [ ]:
from tests import run_transformer_tests
# run_transformer_tests(TransformerBlock, TransformerLM)